# 7-6_2 Transformerモデルを今の書き方で確認する

`7-6_Transformer.ipynb` は、分類用 Transformer の各層を実装し、`utils/dataloader.py` のミニバッチで入出力の形を確認する。このファイルは、層の計算を 7-6 と同じままにし、ミニバッチの作り方だけを今の `Dataset` と `DataLoader` にする。

`batch.Text[0]` に当たるのが `text`、`TEXT.vocab.vectors` に当たるのが `vectors` である。バッチサイズは 7-6 と同じ 24 である。


# 7-6_2 学習目標

1.	Transformerのモジュール構成を理解する
2.	LSTMやRNNを使用せずAttentionで自然言語処理が可能な理由を理解する
3.	Transformerを実装できるようになる


# 事前準備

7-5 で作った次のファイルを使う。

- `data/IMDb_train.tsv`
- `data/IMDb_test.tsv`
- `data/wiki-news-300d-1M.vec`

fastText の読み込みには gensim 3.8.3 を使う。パッケージは Docker の JupyterLab（http://127.0.0.1:8887/lab）に入っている。


In [ ]:
import math
import random
import re
import string
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from gensim.models import KeyedVectors
from torch.utils.data import DataLoader, Dataset


In [ ]:
# Setup seeds
torch.manual_seed(1234)
np.random.seed(1234)
random.seed(1234)


# DatasetとDataLoaderを作成

7-6 は次の関数で、長さ 256、バッチサイズ 24 の DataLoader を受け取っていた。

```python
from utils.dataloader import get_IMDb_DataLoaders_and_TEXT
train_dl, val_dl, test_dl, TEXT = get_IMDb_DataLoaders_and_TEXT(
    max_length=256, batch_size=24)
```

動作確認では `batch.Text[0]` をモデルへ入れ、`TEXT.vocab.vectors` を `Embedder` に渡している。ここでは 7-5_2 と同じ手順で、その2つを作る。訓練データはシャッフルし、検証データとテストデータはデータセットの順のまま取り出す。


In [ ]:
def preprocessing_text(text):
    text = re.sub("<br />", "", text)
    for p in string.punctuation:
        if (p == ".") or (p == ","):
            continue
        text = text.replace(p, " ")
    text = text.replace(".", " . ")
    text = text.replace(",", " , ")
    return text


def tokenizer_with_preprocessing(text):
    return preprocessing_text(text).strip().split()


print(tokenizer_with_preprocessing("I like cats."))


In [ ]:
class TextDataset(Dataset):
    """tsv の1行を (単語リスト, ラベル) として持つ。"""

    def __init__(self, samples):
        self.samples = list(samples)

    @classmethod
    def from_tsv(cls, path, tokenizer):
        samples = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                text, label, *_rest = line.rstrip("\n").split("\t")
                tokens = [token.lower() for token in tokenizer(text)]
                samples.append((tokens, label))
        return cls(samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


train_val_ds = TextDataset.from_tsv("./data/IMDb_train.tsv", tokenizer_with_preprocessing)
test_ds = TextDataset.from_tsv("./data/IMDb_test.tsv", tokenizer_with_preprocessing)
print("訓練および検証のデータ数", len(train_val_ds))
print("テストのデータ数", len(test_ds))


In [ ]:
def split_train_val(dataset, split_ratio=0.8, seed=1234):
    random.seed(seed)
    order = random.sample(range(len(dataset)), len(dataset))
    train_len = int(round(split_ratio * len(dataset)))
    train_samples = [dataset[index] for index in order[:train_len]]
    val_samples = [dataset[index] for index in order[train_len:]]
    return TextDataset(train_samples), TextDataset(val_samples)


train_ds, val_ds = split_train_val(train_val_ds)
print("訓練データの数", len(train_ds))
print("検証データの数", len(val_ds))


In [ ]:
english_fasttext = KeyedVectors.load_word2vec_format(
    "data/wiki-news-300d-1M.vec", binary=False)


def build_vocab(dataset, min_freq=10):
    freqs = Counter()
    for tokens, _label in dataset:
        freqs.update(tokens)
    specials = ["<unk>", "<pad>", "<cls>", "<eos>"]
    for token in specials:
        if token in freqs:
            del freqs[token]
    words = [word for word, freq in freqs.items() if freq >= min_freq]
    words.sort(key=lambda word: (-freqs[word], word))
    itos = specials + words
    stoi = {word: index for index, word in enumerate(itos)}
    return stoi, itos


def build_vector_table(keyed_vectors, stoi):
    table = torch.zeros(len(stoi), keyed_vectors.vector_size)
    for word, index in stoi.items():
        if word in keyed_vectors:
            table[index] = torch.tensor(keyed_vectors[word].copy(), dtype=table.dtype)
    return table


stoi, itos = build_vocab(train_ds, min_freq=10)
vectors = build_vector_table(english_fasttext, stoi)
del english_fasttext
print("語彙ベクトルのサイズ:", vectors.shape)


In [ ]:
max_length = 256
content_length = max_length - 2  # <cls> と <eos>
pad_id = stoi["<pad>"]
unk_id = stoi["<unk>"]
cls_id = stoi["<cls>"]
eos_id = stoi["<eos>"]


def collate_batch(batch):
    text_batch = []
    lengths = []
    labels = []
    for tokens, label in batch:
        ids = [stoi.get(token, unk_id) for token in tokens[:content_length]]
        ids = [cls_id] + ids + [eos_id]
        length = len(ids)
        ids += [pad_id] * (max_length - len(ids))
        text_batch.append(torch.tensor(ids, dtype=torch.long))
        lengths.append(length)
        labels.append(int(label))
    text = torch.stack(text_batch)
    lengths = torch.tensor(lengths, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.long)
    return text, lengths, labels


train_dl = DataLoader(train_ds, batch_size=24, shuffle=True, collate_fn=collate_batch)
val_dl = DataLoader(val_ds, batch_size=24, shuffle=False, collate_fn=collate_batch)
test_dl = DataLoader(test_ds, batch_size=24, shuffle=False, collate_fn=collate_batch)

text, lengths, labels = next(iter(train_dl))
print("文章の ID:", text.shape)
print("ラベル:", labels.shape)


# Embedder

単語 ID を、学習済みベクトルへ変換する。`freeze=True` なので、この層の重みは学習で更新しない。


In [ ]:
class Embedder(nn.Module):
    '''idで示されている単語をベクトルに変換します'''

    def __init__(self, text_embedding_vectors):
        super(Embedder, self).__init__()

        self.embeddings = nn.Embedding.from_pretrained(
            embeddings=text_embedding_vectors, freeze=True)
        # freeze=Trueによりバックプロパゲーションで更新されず変化しなくなります

    def forward(self, x):
        x_vec = self.embeddings(x)

        return x_vec


# 動作確認

7-6 では `x = batch.Text[0]`、`Embedder(TEXT.vocab.vectors)` だった。ここでは `text` と `vectors` を使う。


In [ ]:
# 動作確認
net1 = Embedder(vectors)

x = text  # 7-6 の batch.Text[0]
x1 = net1(x)

print("入力のテンソルサイズ：", x.shape)
print("出力のテンソルサイズ：", x1.shape)


# PositionalEncoder

単語の位置を表すベクトルを、単語ベクトルに足す。


In [ ]:
class PositionalEncoder(nn.Module):
    '''入力された単語の位置を示すベクトル情報を付加する'''

    def __init__(self, d_model=300, max_seq_len=256):
        super().__init__()

        self.d_model = d_model  # 単語ベクトルの次元数

        # 単語の順番（pos）と埋め込みベクトルの次元の位置（i）によって一意に定まる値の表をpeとして作成
        pe = torch.zeros(max_seq_len, d_model)

        # GPUが使える場合はGPUへ送る、ここでは省略。実際に学習時には使用する
        # device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        # pe = pe.to(device)

        for pos in range(max_seq_len):
            for i in range(0, d_model, 2):
                pe[pos, i] = math.sin(pos / (10000 ** ((2 * i)/d_model)))
                
                # 誤植修正_200510 #79
                # pe[pos, i + 1] = math.cos(pos /
                #                          (10000 ** ((2 * (i + 1))/d_model)))
                pe[pos, i + 1] = math.cos(pos /
                                          (10000 ** ((2 * i)/d_model)))

        # 表peの先頭に、ミニバッチ次元となる次元を足す
        self.pe = pe.unsqueeze(0)

        # 勾配を計算しないようにする
        self.pe.requires_grad = False

    def forward(self, x):

        # 入力xとPositonal Encodingを足し算する
        # xがpeよりも小さいので、大きくする
        ret = math.sqrt(self.d_model)*x + self.pe
        return ret


In [ ]:
# 動作確認
net1 = Embedder(vectors)
net2 = PositionalEncoder(d_model=300, max_seq_len=256)

x = text
x1 = net1(x)
x2 = net2(x1)

print("入力のテンソルサイズ：", x1.shape)
print("出力のテンソルサイズ：", x2.shape)


# Attention

7-6 と同じく、マルチヘッドにはせず、単一の Attention で計算する。


In [ ]:
class Attention(nn.Module):
    '''Transformerは本当はマルチヘッドAttentionですが、
    分かりやすさを優先しシングルAttentionで実装します'''

    def __init__(self, d_model=300):
        super().__init__()

        # SAGANでは1dConvを使用したが、今回は全結合層で特徴量を変換する
        self.q_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)

        # 出力時に使用する全結合層
        self.out = nn.Linear(d_model, d_model)

        # Attentionの大きさ調整の変数
        self.d_k = d_model

    def forward(self, q, k, v, mask):
        # 全結合層で特徴量を変換
        k = self.k_linear(k)
        q = self.q_linear(q)
        v = self.v_linear(v)

        # Attentionの値を計算する
        # 各値を足し算すると大きくなりすぎるので、root(d_k)で割って調整
        weights = torch.matmul(q, k.transpose(1, 2)) / math.sqrt(self.d_k)

        # ここでmaskを計算
        mask = mask.unsqueeze(1)
        weights = weights.masked_fill(mask == 0, -1e9)

        # softmaxで規格化をする
        normlized_weights = F.softmax(weights, dim=-1)

        # AttentionをValueとかけ算
        output = torch.matmul(normlized_weights, v)

        # 全結合層で特徴量を変換
        output = self.out(output)

        return output, normlized_weights


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=1024, dropout=0.1):
        '''Attention層から出力を単純に全結合層2つで特徴量を変換するだけのユニットです'''
        super().__init__()

        self.linear_1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = self.linear_1(x)
        x = self.dropout(F.relu(x))
        x = self.linear_2(x)
        return x


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()

        # LayerNormalization層
        # https://pytorch.org/docs/stable/nn.html?highlight=layernorm
        self.norm_1 = nn.LayerNorm(d_model)
        self.norm_2 = nn.LayerNorm(d_model)

        # Attention層
        self.attn = Attention(d_model)

        # Attentionのあとの全結合層2つ
        self.ff = FeedForward(d_model)

        # Dropout
        self.dropout_1 = nn.Dropout(dropout)
        self.dropout_2 = nn.Dropout(dropout)

    def forward(self, x, mask):
        # 正規化とAttention
        x_normlized = self.norm_1(x)
        output, normlized_weights = self.attn(
            x_normlized, x_normlized, x_normlized, mask)
        
        x2 = x + self.dropout_1(output)

        # 正規化と全結合層
        x_normlized2 = self.norm_2(x2)
        output = x2 + self.dropout_2(self.ff(x_normlized2))

        return output, normlized_weights


# 動作確認

`<pad>` の位置だけ False にしたマスクを TransformerBlock に渡す。7-6 ではこの ID を `1` と書いていた。`<pad>` の ID は 1 である。


In [ ]:
# 動作確認
net1 = Embedder(vectors)
net2 = PositionalEncoder(d_model=300, max_seq_len=256)
net3 = TransformerBlock(d_model=300)

x = text
input_mask = (x != pad_id)
print(input_mask[0])

x1 = net1(x)
x2 = net2(x1)
x3, normlized_weights = net3(x2, input_mask)

print("入力のテンソルサイズ：", x2.shape)
print("出力のテンソルサイズ：", x3.shape)
print("Attentionのサイズ：", normlized_weights.shape)


# ClassificationHead

各文の先頭、つまり `<cls>` の特徴量から、ポジティブとネガティブの 2 値を出す。


In [ ]:
class ClassificationHead(nn.Module):
    '''Transformer_Blockの出力を使用し、最後にクラス分類させる'''

    def __init__(self, d_model=300, output_dim=2):
        super().__init__()

        # 全結合層
        self.linear = nn.Linear(d_model, output_dim)  # output_dimはポジ・ネガの2つ

        # 重み初期化処理
        nn.init.normal_(self.linear.weight, std=0.02)
        nn.init.normal_(self.linear.bias, 0)

    def forward(self, x):
        x0 = x[:, 0, :]  # 各ミニバッチの各文の先頭の単語の特徴量（300次元）を取り出す
        out = self.linear(x0)

        return out


In [ ]:
# 動作確認
text, lengths, labels = next(iter(train_dl))

net1 = Embedder(vectors)
net2 = PositionalEncoder(d_model=300, max_seq_len=256)
net3 = TransformerBlock(d_model=300)
net4 = ClassificationHead(output_dim=2, d_model=300)

x = text
input_mask = (x != pad_id)
x1 = net1(x)
x2 = net2(x1)
x3, normlized_weights = net3(x2, input_mask)
x4 = net4(x3)

print("入力のテンソルサイズ：", x3.shape)
print("出力のテンソルサイズ：", x4.shape)


# TransformerClassification

ここまでの層を、2 段の TransformerBlock と分類ヘッドにつなぐ。


In [ ]:
# 最終的なTransformerモデルのクラス


class TransformerClassification(nn.Module):
    '''Transformerでクラス分類させる'''

    def __init__(self, text_embedding_vectors, d_model=300, max_seq_len=256, output_dim=2):
        super().__init__()

        # モデル構築
        self.net1 = Embedder(text_embedding_vectors)
        self.net2 = PositionalEncoder(d_model=d_model, max_seq_len=max_seq_len)
        self.net3_1 = TransformerBlock(d_model=d_model)
        self.net3_2 = TransformerBlock(d_model=d_model)
        self.net4 = ClassificationHead(output_dim=output_dim, d_model=d_model)

    def forward(self, x, mask):
        x1 = self.net1(x)  # 単語をベクトルに
        x2 = self.net2(x1)  # Positon情報を足し算
        x3_1, normlized_weights_1 = self.net3_1(
            x2, mask)  # Self-Attentionで特徴量を変換
        x3_2, normlized_weights_2 = self.net3_2(
            x3_1, mask)  # Self-Attentionで特徴量を変換
        x4 = self.net4(x3_2)  # 最終出力の0単語目を使用して、分類0-1のスカラーを出力
        return x4, normlized_weights_1, normlized_weights_2


In [ ]:
# 動作確認
text, lengths, labels = next(iter(train_dl))

net = TransformerClassification(
    text_embedding_vectors=vectors, d_model=300, max_seq_len=256, output_dim=2)

x = text
input_mask = (x != pad_id)
out, normlized_weights_1, normlized_weights_2 = net(x, input_mask)

print("出力のテンソルサイズ：", out.shape)
print("出力テンソルのsoftmax：", F.softmax(out, dim=1))


ここまでのクラスは、`utils/transformer.py` に入っているものと同じ計算である。7-7 はそのファイルを読み込む。このノートは動作確認用なので、`utils/transformer.py` は書き換えない。

以上
